# P13 – Spektar snimke: duljina odsječka, prozor i nadopuna nulama

Bilježnica uz predavanje P13 i **poglavlje 13 udžbenika *Signali i sustavi*** (Fourierova
transformacija diskretnih signala). Iz snimke uzmemo odsječak od $N$ uzoraka, pomnožimo ga
prozorom i izračunamo mu jednostrani amplitudni spektar preko DFT-a (u praksi FFT). Klizačima se
biraju duljina odsječka $N$, prozor (pravokutni ili Hannov) i nadopuna nulama, a graf pokazuje kako
o njima ovise frekvencijska razlučivost i curenje spektra. Zadana „snimka” je sintetička: ton dviju
žica ugođenih gotovo na isti ton, s harmonicima, tihim brujanjem i malo šuma. Umjesto nje možete
učitati vlastitu snimku (WAV), npr. zviždanje, pjevani samoglasnik, ton gitare ili zujanje
ventilatora.

**Kako pokrenuti.** Ćelije pokrenite redom (Shift + Enter) ili izbornikom *Run → Run All
Cells*. Instalaciju Pythona i JupyterLaba opisuje `README.md` repozitorija. Bilježnica radi bez
interneta, a zadanu snimku sama sintetizira. Za slušanje trebaju zvučnici ili slušalice; bez njih
rade grafovi i ispis. Ako se klizači ne prikažu, funkciju pokusa pozovite izravno (odjeljak 4).

In [ ]:
import html
import os
import warnings
from math import gcd
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy import signal
from scipy.io import wavfile
from scipy.signal import resample_poly
from IPython.display import Audio, display
import ipywidgets as widgets

MOJ_ZVUK = Path("moj_zvuk")            # vlastite snimke (nije u gitu)
BOJA = "#840B55"                       # bordo: Hannov prozor i odsječak pomnožen prozorom
SIVA, TAMNA = "#5B6770", "#222222"
# Krivulje se razlikuju i vrstom crte ili oznakom, pa su čitljive i crno-bijelo.

# decimalni zarez i pravi minus na osima
zarez = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})

## Postavke: zadana ili vlastita snimka

Bilježnica zadano analizira sintetičku snimku koju sama izračuna (odjeljak 2). Za **vlastitu
snimku** upišite njezinu putanju između navodnika u ćeliji ispod (`SNIMKA`), pa pokrenite sve
ćelije (*Run → Run All Cells*). Prazni navodnici `""` znače zadanu snimku. Druga mogućnost je gumb
za prijenos datoteke u idućoj ćeliji.

Bilježnica čita samo **WAV** (mono ili stereo, 8, 16, 24 ili 32 bita, bilo koja frekvencija
uzorkovanja). Snimku iz mobitela (M4A, MP3, OGG, FLAC …) najprije pretvorite u WAV, npr.
besplatnim programom **Audacity**: *File → Open* (otvorite snimku), zatim *File → Export
Audio…* (u starijim inačicama *File → Export → Export as WAV*), vrsta datoteke *WAV*,
kodiranje *Signed 16-bit PCM*. Ako je instaliran program ffmpeg, isto radi naredba
`ffmpeg -i snimka.m4a snimka.wav`.

Ostale postavke: frekvencija uzorkovanja analize `FS` (vlastita snimka se na nju preuzorkuje),
početak odsječka, raspon frekvencija na grafu, sastav zadane snimke i mreže klizača.

In [ ]:
# ===== POSTAVKE (mijenjajte po želji, pa Run → Run All Cells) =====
# Vlastita snimka: putanja relativna prema mapi bilježnice. Prazno ("") = zadana sintetička.
SNIMKA = ""              # npr. "moj_zvuk/zvizduk.wav"
FS = 16000               # Hz: frekvencija uzorkovanja f_s analize (vlastita snimka se preuzorkuje)
POCETAK_S = 0.25         # s: odsječak počinje toliko nakon početka zvuka
MAX_SNIMKA_S = 20        # s: dulja vlastita snimka skraćuje se
F_PRIKAZ = 2000          # Hz: gornja granica spektra na grafu u decibelima
ZOOM_HZ = 25             # Hz: pola širine uvećanog dijela spektra oko najvišeg vrha

# Zadana sintetička snimka: dvije žice ugođene gotovo na isti ton, s harmonicima na
# frekvencijama k·F0 i k·(F0 + RASTIMANOST), tiho brujanje i malo šuma
TRAJANJE = 3.0           # s
F0 = 220                 # Hz: osnovna frekvencija prve žice
RASTIMANOST = 4          # Hz: osnovna frekvencija druge žice viša je za toliko
HARMONICI = (0.2, 0.1, 0.06, 0.04, 0.025, 0.016, 0.01, 0.006)   # amplitude C_k prve žice
DRUGA_ZICA = 0.8         # amplitude druge žice su DRUGA_ZICA · C_k
F_SLABA = 290            # Hz: tiho brujanje (npr. ventilator)
C_SLABA = 0.001          # amplituda brujanja
SUM = 0.0005             # standardna devijacija šuma
SJEME = 6                # sjeme generatora slučajnih brojeva (fazni kutovi i šum)
# SJEME i fazni kutovi su izabrani tako da prikaz razlučivanja u odjeljcima 4 i 5 odgovara
# pravilu o razlučivanju iz poglavlja 13 (uz neka druga sjemena Hannov prozor slučajno pokaže
# dva lažna vrha); to provjerava test. Promjena SJEME, F0, RASTIMANOST ili HARMONICI može
# promijeniti taj prikaz.

# Klizači
N_IZBOR = (256, 512, 1024, 2048, 4096, 8192, 16384, 32768)     # duljina odsječka N
N_POCETNI = 2048
PROZORI = ("pravokutni", "Hannov")
PROZOR_POCETNI = "pravokutni"
NADOPUNA_IZBOR = (1, 2, 4, 8, 16)    # duljina DFT-a je nadopuna · N (1: bez nadopune)
NADOPUNA_POCETNA = 1

# Iz knjige (poglavlje 13, pravilo o pravokutnom i Hannovu prozoru): najviši bočni režanj
# prema glavnom [dB] uz N = PROZOR_N_KNJIGA (odjeljak 3)
PROZOR_N_KNJIGA = 64
BOCNI_PRAVOKUTNI_KNJIGA = -13.3
BOCNI_HANN_KNJIGA = -31.5

# Za automatske testove: prazno polje preuzima vrijednost varijable okoline P13_SNIMKA, ako
# je postavljena. Na predavanju to ne treba: upišite putanju gore.
SNIMKA = SNIMKA or os.environ.get("P13_SNIMKA", "")

In [ ]:
# Prijenos snimke gumbom (JupyterLab; radi i bez interneta). Gumb otvori odabir datoteke.
# Prenesena datoteka sprema se u moj_zvuk/ i zamjenjuje SNIMKA iz ćelije iznad. Nakon
# prijenosa ponovno pokrenite sve ćelije (Run → Run All Cells): gumb pamti datoteku do
# ponovnog pokretanja jezgre (Kernel → Restart Kernel) ili do naredbe gumb_snimka.value = ().
# Bez prijenosa ćelija ništa ne čeka. Datoteke veće od oko 10 MB radije kopirajte u moj_zvuk/
# i upišite putanju gore.


def spremi_prijenos(preneseno, mapa=MOJ_ZVUK):
    # zadnja datoteka prenesena gumbom -> spremljena u mapu; vraća njezinu putanju ili ""
    if not preneseno:
        return ""
    datoteka = preneseno[-1]                   # ipywidgets 8: n-torka rječnika
    mapa = Path(mapa)
    mapa.mkdir(parents=True, exist_ok=True)
    put = mapa / Path(datoteka["name"]).name    # samo ime, bez mape iz preglednika
    put.write_bytes(bytes(datoteka["content"]))
    return put.as_posix()


if "gumb_snimka" not in globals():              # isti gumb i nakon Run All Cells
    gumb_snimka = widgets.FileUpload(accept=".wav", description="snimka (WAV)")
    stanje = widgets.HTML("nije prenesena nijedna datoteka")

    def _preneseno(_promjena):
        if gumb_snimka.value:
            ime = html.escape(gumb_snimka.value[-1]["name"])
            stanje.value = f"preneseno: {ime}; sada <b>Run → Run All Cells</b>"

    gumb_snimka.observe(_preneseno, "value")

display(widgets.HBox([gumb_snimka, stanje]))
SNIMKA = spremi_prijenos(gumb_snimka.value) or SNIMKA

### Kako snimiti vlastiti zvuk

1. Izaberite zvuk koji neko vrijeme traje jednako: zviždanje, pjevani samoglasnik („aaa”), ton
   žice gitare, zujanje ventilatora, usisavača ili hladnjaka.
2. Mobitel držite tridesetak centimetara od izvora. Za vrijeme snimanja u prostoriji mora biti
   tiho. Snimajte nekoliko sekundi.
3. Snimku pretvorite u WAV (vidi gore) i spremite je u mapu `moj_zvuk/`, npr. kao
   `moj_zvuk/zvizduk.wav`. U postavkama upišite `SNIMKA = "moj_zvuk/zvizduk.wav"`.

Bilježnica snimku pripremi sama i ispiše svaku promjenu: stereo pretvori u mono, preuzorkuje je
na `FS`, odreže tišinu na početku, skrati je na `MAX_SNIMKA_S` i normira vrh na 0,9, pa su
amplitude u spektru relativne. Ako je snimka prekratka za najveći $N$ s klizača, klizač nudi samo
$N$ koji stanu u snimku.

**Licenca.** Mapa `moj_zvuk/` nije u gitu, pa vlastite snimke ostaju samo na vašem računalu.
U repozitorij (mapa za zapise uz bilježnicu, s izvorom i licencom u `LICENSE-sadrzaj.md`) snimka
smije samo ako ima licencu koja dopušta objavu (CC0, CC BY 4.0 ili CC BY-SA 4.0) i ako se na
njoj ne čuju drugi ljudi bez njihova pristanka.

## 1. Spektar odsječka snimke

Snimka su uzorci $x[n]$ uz frekvenciju uzorkovanja $f_s$. Za spektar uzmemo odsječak od $N$
uzoraka, trajanja $T = N/f_s$, pomnožimo ga prozorom $w[n]$ i izračunamo DFT:

$$X_w[k] = \sum_{n=0}^{N-1} w[n]\,x[n]\,e^{-j2\pi kn/N}, \qquad k = 0, \dots, N-1 .$$

Bin $k$ (frekvencijski uzorak DFT-a, engl. *bin*) je frekvencija $f_k = k f_s/N$, a razmak
binova $\Delta f = f_s/N = 1/T$ je **frekvencijska razlučivost**. Ovisi samo o trajanju
odsječka. Crtamo binove od $0$ do Nyquistove frekvencije $f_s/2$, kao **jednostrani
amplitudni spektar** (definicija u
poglavlju 13): $A[k] = \frac{2}{\sum w[n]}\,|X_w[k]|$ za $0 < k < N/2$, a za rubne binove
$k = 0$ i $k = N/2$ skala je $1/\sum w[n]$. Bez množenja (pravokutni prozor, $w[n] = 1$) je
$\sum w[n] = N$ i skala je $2/N$. Sinusoida amplitude $C$ čija frekvencija leži na binu daje
$A = C$. Frekvencija koja ne leži na binu **curi**: vrh je niži, a ostatak se razlije po svim
binovima (teorem o curenju spektra).

**Hannov prozor** $w[n] = 0{,}5 - 0{,}5\cos(2\pi n/N)$ glatko spušta odsječak na nulu na oba
kraja. Curi mnogo manje od pravokutnog, ali mu je vrh dvostruko širi (odjeljak 3).

**Nadopuna nulama.** Odsječku od $N$ uzoraka dopišemo nule do duljine $L N$ i računamo DFT od $LN$
točaka (ovdje $L$ = `nadopuna`). To samo gušće uzorkuje isti DTFT $X_w(e^{j\varOmega})$: krivulja
je glađa, a vrh ostaje iste širine. Skala ostaje $2/\sum w[n]$ jer se zbroj prozora ne mijenja.

Na grafu u decibelima crtamo $20\log_{10} A[k]$. Funkcije u ćeliji ispod računaju prozor,
spektar, zadanu snimku i ispis; spektar je onaj iz postupka za spektar snimke u poglavlju 13.

In [ ]:
def broj(v, decimala=0):
    # broj s decimalnim zarezom i pravim minusom (−13,3); −0 piše se 0
    tekst = f"{v:.{decimala}f}"
    if float(tekst) == 0:
        tekst = tekst.lstrip("-")
    return tekst.replace(".", ",").replace("-", "−")


def hann(N):
    # Hannov prozor w[n] = 0,5 − 0,5·cos(2πn/N), n = 0, …, N − 1 (periodični oblik za DFT)
    n = np.arange(N)
    return 0.5 - 0.5 * np.cos(2 * np.pi * n / N)


def prozor_niz(ime, N):
    # prozor w[n] duljine N: pravokutni (w[n] = 1) ili Hannov
    if ime == "pravokutni":
        return np.ones(N)
    if ime == "Hannov":
        return hann(N)
    raise ValueError(f"nepoznat prozor „{ime}”; izaberite jedan od {PROZORI}")


def spektar(x, prozor="pravokutni", nadopuna=1):
    # jednostrani amplitudni spektar odsječka x pomnoženog prozorom: frekvencije točaka
    # k·f_s/N_DFT i amplitude 2|X_w[k]|/Σw (rubne točke 1/Σw), uz nadopunu nulama na
    # N_DFT = nadopuna·N (nadopuna = 1: binovi f_k = k·f_s/N)
    x = np.asarray(x, dtype=float)
    if int(nadopuna) != nadopuna or nadopuna < 1:
        raise ValueError("nadopuna mora biti cijeli broj, najmanje 1 (1: bez nadopune)")
    w = prozor_niz(prozor, len(x))
    n_dft = int(nadopuna) * len(x)
    X = np.fft.rfft(w * x, n_dft)
    A = 2 * np.abs(X) / w.sum()
    A[0] /= 2
    if n_dft % 2 == 0:
        A[-1] /= 2
    return np.arange(len(A)) * FS / n_dft, A


def sinteza():
    # zadana sintetička snimka uz FS: dvije žice s harmonicima, tiho brujanje i šum
    rng = np.random.default_rng(SJEME)
    t = np.arange(int(TRAJANJE * FS)) / FS
    s = np.zeros(len(t))
    for k, C in enumerate(HARMONICI, start=1):
        s += C * np.cos(2 * np.pi * k * F0 * t + rng.uniform(0, 2 * np.pi))
        s += DRUGA_ZICA * C * np.cos(2 * np.pi * k * (F0 + RASTIMANOST) * t
                                     + rng.uniform(0, 2 * np.pi))
    s += C_SLABA * np.cos(2 * np.pi * F_SLABA * t) + SUM * rng.standard_normal(len(t))
    rub = int(0.02 * FS)                       # 20 ms mekog početka i kraja (bez klika)
    s[:rub] *= np.linspace(0, 1, rub)
    s[-rub:] *= np.linspace(1, 0, rub)
    return s


def pocetak_odsjecka(duljina):
    # prvi uzorak odsječka: POCETAK_S nakon početka, ako snimka to dopušta, inače od početka
    p = int(POCETAK_S * FS)
    return p if p + N_IZBOR[0] <= duljina else 0


def moguci_n(duljina, pocetak):
    # duljine odsječka iz N_IZBOR koje stanu u snimku od prvog uzorka odsječka
    return tuple(N for N in N_IZBOR if pocetak + N <= duljina)


def vrh(f, A, f_max=F_PRIKAZ, f_min=20):
    # najviši vrh spektra od f_min (donja granica sluha; ispod nje su npr. vjetar i pomak
    # mikrofona) do f_max: (frekvencija, amplituda)
    i = np.flatnonzero((f >= f_min) & (f <= f_max))
    j = i[np.argmax(A[i])]
    return f[j], A[j]


def bocna_latica_db(w, gustoca=1024):
    # najviši bočni režanj |W_w(e^{jΩ})| prozora w prema glavnom, u dB: DTFT na gustoj mreži
    # (DFT nadopunjen nulama); glavni režanj završava u prvom minimumu
    W = np.abs(np.fft.rfft(w, gustoca * len(w)))
    i = int(np.argmax(np.diff(W) > 0))
    return 20 * np.log10(W[i:].max() / W[0])


def opis(x, prozor, nadopuna):
    # retci ispisa za odsječak x (N uzoraka) uz prozor i nadopunu nulama
    N = len(x)
    w = prozor_niz(prozor, N)
    f, A = spektar(x, prozor, nadopuna)
    f_v, A_v = vrh(f, A)
    binova = 2 if prozor == "pravokutni" else 4
    retci = [f"odsječak: N = {N} uzoraka, T = N/f_s = {broj(N / FS, 3)} s (f_s = {FS} Hz)",
             f"binovi f_k = k·f_s/N, razlučivost Δf = f_s/N = 1/T = {broj(FS / N, 3)} Hz",
             f"prozor {prozor}: Σw = {broj(w.sum())}, skala 2/Σw; glavni režanj od nule do "
             f"nule {binova} bina = {broj(binova * FS / N, 3)} Hz"]
    if nadopuna == 1:
        retci.append("bez nadopune nulama: točke spektra su binovi f_k")
    else:
        retci.append(f"nadopuna nulama na {nadopuna}·N = {nadopuna * N} točaka: točke spektra "
                     f"svakih {broj(FS / (nadopuna * N), 3)} Hz, a razlučivost ostaje Δf")
    retci.append(f"najviši vrh do {broj(F_PRIKAZ)} Hz: f = {broj(f_v, 1)} Hz, "
                 f"A = {broj(A_v, 2)}")
    return retci


def opis_snimke(s, ime, pocetak, n_moguci, fs_izvorno=None):
    # retci ispisa za učitanu snimku (s je već uz FS; fs_izvorno je frekvencija uzorkovanja
    # datoteke); javlja grešku ako ni najmanji N ne stane u snimku
    if not n_moguci:
        raise ValueError(f"{ime}: snimka je prekratka; treba barem {N_IZBOR[0]} uzoraka "
                         f"({broj(N_IZBOR[0] / FS, 3)} s) zvuka uz f_s = {FS} Hz.")
    if fs_izvorno is None or fs_izvorno == FS:
        uzorkovanje = f"f_s = {FS} Hz"
    else:
        uzorkovanje = f"izvorno f_s = {fs_izvorno} Hz, analiza uz f_s = {FS} Hz"
    retci = [f"{ime}: {uzorkovanje}, {len(s)} uzoraka, trajanje {broj(len(s) / FS, 2)} s",
             f"odsječak počinje u t = {broj(pocetak / FS, 2)} s; klizač N od {n_moguci[0]} "
             f"do {n_moguci[-1]}"]
    if n_moguci[-1] < N_IZBOR[-1]:
        retci.append(f"za N = {N_IZBOR[-1]} snimka bi trebala trajati barem "
                     f"{broj((pocetak + N_IZBOR[-1]) / FS, 2)} s")
    return retci


def ovojnica(f, y, tocaka=3000):
    # za crtanje: najviše oko tocaka točaka, svaka najveća vrijednost svog dijela osi (ovojnica
    # bočnih režnjeva i vrhovi na pravim frekvencijama); ispis koristi sve točke
    if len(y) <= tocaka:
        return f, y
    k = -(-len(y) // tocaka)
    Y = np.concatenate([y, np.full((-len(y)) % k, -np.inf)]).reshape(-1, k)
    j = Y.argmax(axis=1) + k * np.arange(len(Y))
    return f[j], y[j]


def tablica_prozora():
    # najviši bočni režanj i Σw pravokutnog i Hannova prozora uz N = PROZOR_N_KNJIGA
    N = PROZOR_N_KNJIGA
    retci = [f"N = {N}       najviši bočni režanj / dB    knjiga / dB      Σw"]
    for ime, knjiga in (("pravokutni", BOCNI_PRAVOKUTNI_KNJIGA),
                        ("Hannov", BOCNI_HANN_KNJIGA)):
        w = prozor_niz(ime, N)
        retci.append(f"{ime:<11} {broj(bocna_latica_db(w), 1):>26} {broj(knjiga, 1):>14}"
                     f" {broj(w.sum()):>7}")
    return retci

In [ ]:
# Učitavanje vlastite snimke (WAV), kao u bilježnici P08: 8, 16, 24 i 32 bita, stereo u
# mono, preuzorkovanje na FS; za datoteku koja ne postoji, nije WAV ili je tiha javlja grešku
# s uputom. Svaku promjenu snimke ispiše.
PRAG_TISINE = 0.03       # tišina na početku: ispod 3 % vršne vrijednosti (−30 dB)
UPUTA_WAV = ("Bilježnica čita samo WAV. Pretvorite zapis u WAV, npr. u Audacityju: "
             "File → Open, zatim File → Export Audio… (vrsta WAV, Signed 16-bit PCM).")


def zarez2(v):
    return f"{v:.2f}".replace(".", ",")


def nadji(ime, mapa=MOJ_ZVUK):
    # putanja snimke: ime s mapom (npr. "moj_zvuk/zvizduk.wav") ili samo ime u mapi
    put = Path(ime).expanduser()
    kandidati = [put] if put.parent != Path(".") else list(dict.fromkeys(
        [Path(mapa) / put, MOJ_ZVUK / put, put]))
    for k in kandidati:
        if k.is_file():
            return k
    trazeno = ", ".join(k.as_posix() for k in kandidati)
    raise FileNotFoundError(f"Nema datoteke „{ime}” (traženo: {trazeno}). Putanja je "
                            "relativna prema mapi bilježnice, npr. \"moj_zvuk/zvizduk.wav\".")


def preuzorkuj(s, fs_zapisa, fs, ime=""):
    # preuzorkovanje s fs_zapisa na fs (polifazni filtar, scipy.signal.resample_poly)
    if fs_zapisa == fs:
        return s
    d = gcd(fs, fs_zapisa)
    print(f"{ime}: preuzorkovano s {fs_zapisa} Hz na {fs} Hz")
    return resample_poly(s, fs // d, fs_zapisa // d)


def ucitaj(ime, fs=None, mapa=MOJ_ZVUK):
    # WAV -> (frekvencija uzorkovanja, mono signal u rasponu −1 … 1); uz zadani fs zapis s
    # drugom frekvencijom uzorkovanja preuzorkuje se na fs
    put = nadji(ime, mapa)
    if put.suffix.lower() != ".wav":
        raise ValueError(f"{put.name}: nije WAV datoteka. {UPUTA_WAV}")
    try:
        with warnings.catch_warnings():        # dodatni podaci u WAV-u s mobitela
            warnings.simplefilter("ignore", wavfile.WavFileWarning)
            fs_zapisa, s = wavfile.read(put)
    except ValueError as e:                    # npr. MP3 preimenovan u .wav, komprimirani WAV
        raise ValueError(f"{put.name}: ne mogu pročitati kao WAV ({e}). {UPUTA_WAV}") from None
    if s.dtype == np.uint8:                    # 8-bitni WAV: 0 … 255, tišina = 128
        s = (s.astype(float) - 128) / 128
    elif np.issubdtype(s.dtype, np.integer):   # 16, 24 (učitan kao 32) i 32 bita
        s = s / np.iinfo(s.dtype).max
    s = np.asarray(s, dtype=float)
    if s.ndim == 2:                            # stereo -> mono
        s = s.mean(axis=1)
    if s.size == 0 or not np.max(np.abs(s)) > 1e-3:
        raise ValueError(f"{put.name}: zapis je prazan ili tih (vrh ispod −60 dB). "
                         "Provjerite snimku, npr. u Audacityju.")
    if fs is not None:
        s = preuzorkuj(s, fs_zapisa, fs, put.name)
        fs_zapisa = fs
    return fs_zapisa, s


def pripremi(s, fs):
    # vlastita snimka: bez tišine na početku, najviše MAX_SNIMKA_S sekundi, vrh 0,9
    pocetak = int(np.flatnonzero(np.abs(s) > PRAG_TISINE * np.max(np.abs(s)))[0])
    if pocetak > int(0.05 * fs):
        print(f"snimka: odrezano {zarez2(pocetak / fs)} s tišine na početku")
        s = s[pocetak:]
    if len(s) > MAX_SNIMKA_S * fs:
        print(f"snimka: traje {zarez2(len(s) / fs)} s; uzeto prvih {MAX_SNIMKA_S} s "
              "(MAX_SNIMKA_S u postavkama)")
        s = s[:int(MAX_SNIMKA_S * fs)]
    print("snimka: vrh normiran na 0,9 (amplitude u spektru su relativne)")
    return 0.9 * s / np.max(np.abs(s))

## 2. Snimka

Ćelija ispod učita vlastitu snimku ili izračuna zadanu i ispiše njezine podatke. Zadana snimka
traje 3 s. Svaka žica ima osnovni ton i sedam viših harmonika (poglavlje 9), a druga žica je za
`RASTIMANOST` = 4 Hz viša od prve. Zato se pri slušanju čuju **udari**: glasnoća tona raste i
pada četiri puta u sekundi jer se dvije bliske sinusoide naizmjence zbrajaju i gotovo poništavaju.
Uz to snimka ima tiho brujanje na `F_SLABA` = 290 Hz i malo šuma.

Graf pokazuje cijelu snimku s odsječkom najvećeg $N$ (sivo) i zadanog $N$ (bordo), a ispod njega
prvih 30 ms zadanog odsječka. Odsječak uvijek počinje `POCETAK_S` nakon početka zvuka.

In [ ]:
if SNIMKA:
    fs_izvorno, snimka = ucitaj(SNIMKA)
    ime_snimke = Path(SNIMKA).name
    snimka = pripremi(preuzorkuj(snimka, fs_izvorno, FS, ime_snimke), FS)
else:
    fs_izvorno = FS
    snimka, ime_snimke = sinteza(), "zadana sintetička snimka"
pocetak = pocetak_odsjecka(len(snimka))
n_moguci = moguci_n(len(snimka), pocetak)
for redak in opis_snimke(snimka, ime_snimke, pocetak, n_moguci, fs_izvorno):
    print(redak)

# središte uvećanog dijela spektra: najviši vrh najduljeg odsječka uz Hannov prozor
f_vrh, _ = vrh(*spektar(snimka[pocetak:pocetak + n_moguci[-1]], "Hannov"))
n_zadani = min(N_POCETNI, n_moguci[-1])

t = np.arange(len(snimka)) / FS
fig, (ax, ax_k) = plt.subplots(2, 1, figsize=(11, 5.4))
ax.plot(t, snimka, color=TAMNA, lw=0.4)
ax.axvspan(pocetak / FS, (pocetak + n_moguci[-1]) / FS, color=SIVA, alpha=0.2,
           label=f"odsječak najvećeg $N$ = {n_moguci[-1]}")
ax.axvspan(pocetak / FS, (pocetak + n_zadani) / FS, color=BOJA, alpha=0.35,
           label=f"odsječak zadanog $N$ = {n_zadani}")
ax.set_xlim(0, t[-1])
ax.set_xlabel("vrijeme $t$ / s")
ax.set_ylabel("$x[n]$")
ax.set_title(ime_snimke, loc="left", fontsize=10)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
n30 = min(n_zadani, int(0.03 * FS))
ax_k.plot(1000 * np.arange(n30) / FS, snimka[pocetak:pocetak + n30], color=TAMNA, lw=1)
ax_k.set_xlabel("vrijeme od početka odsječka / ms")
ax_k.set_ylabel("$x[n]$")
ax_k.set_title("prvih 30 ms odsječka", loc="left", fontsize=10)
for a in (ax, ax_k):
    a.xaxis.set_major_formatter(zarez)
    a.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()
print("Cijela snimka:")
display(Audio(snimka, rate=FS))

## 3. Prozori: pravokutni i Hannov

Prema pravilu o pravokutnom i Hannovu prozoru u poglavlju 13 pravokutni prozor ima glavni režanj
širok dva bina (od nule do nule), a bočni režnjevi padaju sporo. Hannov prozor ima dvostruko
širi glavni režanj, četiri bina, ali najviši bočni režanj mu je mnogo niži, a režnjevi padaju
mnogo brže. Uz Hannov prozor je $\sum w[n] = N/2$, pa je skala jednostranog spektra $4/N$.

Ćelija ispod računa najviši bočni režanj obaju prozora iz DTFT-a prozora na gustoj mreži i
uspoređuje ga s knjigom. Lijevo su prozori u vremenu, a desno modul njihova DTFT-a
$|W_w(e^{j\varOmega})|$ u decibelima, prema udaljenosti od vrha u binovima ($N\varOmega/2\pi$).
Spektar jedne sinusoide kroz prozor je upravo ta krivulja, pomaknuta na njezinu frekvenciju
(teorem o curenju spektra): bočni režnjevi su curenje.

In [ ]:
for redak in tablica_prozora():
    print(redak)
for ime, knjiga in (("pravokutni", BOCNI_PRAVOKUTNI_KNJIGA), ("Hannov", BOCNI_HANN_KNJIGA)):
    assert round(bocna_latica_db(prozor_niz(ime, PROZOR_N_KNJIGA)), 1) == knjiga
print("Bilježnica i knjiga daju iste bočne režnjeve.")

N = PROZOR_N_KNJIGA
gustoca = 64
fig, (ax_w, ax_W) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})
for ime, boja, crta in (("pravokutni", TAMNA, "-"), ("Hannov", BOJA, "--")):
    w = prozor_niz(ime, N)
    ax_w.plot(np.arange(N), w, color=boja, ls=crta, lw=1.6, label=ime)
    W = np.abs(np.fft.fft(w, gustoca * N))
    W = np.concatenate([W[-16 * gustoca:], W[:16 * gustoca + 1]])
    pomak = np.arange(-16 * gustoca, 16 * gustoca + 1) / gustoca
    ax_W.plot(pomak, 20 * np.log10(np.maximum(W / W.max(), 1e-6)), color=boja, ls=crta,
              lw=1.4, label=ime)
ax_w.set_xlabel("$n$")
ax_w.set_ylabel("$w[n]$")
ax_w.set_ylim(0, 1.15)
ax_w.set_title(f"prozori, $N$ = {N}", loc="left", fontsize=10)
ax_w.legend(loc="lower center", frameon=True, framealpha=0.9, fontsize=9)
ax_W.set_xlim(-16, 16)
ax_W.set_ylim(-100, 5)
ax_W.set_xlabel("udaljenost od vrha / binovi")
ax_W.set_ylabel("dB prema vrhu")
ax_W.set_title("modul DTFT-a prozora", loc="left", fontsize=10)
ax_W.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
for a in (ax_w, ax_W):
    a.xaxis.set_major_formatter(zarez)
    a.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

## 4. Klizači: duljina odsječka $N$, prozor i nadopuna nulama

Klizač **N** bira duljinu odsječka (potencije broja 2, kao za FFT), gumbi **prozor** pravokutni ili
Hannov prozor, a klizač **nadopuna** koliko puta je DFT dulji od odsječka (1: bez nadopune).

- **Gore:** spektar od 0 do `F_PRIKAZ` u decibelima. Vide se harmonici, a uz pravokutni prozor i
  široki „rep” curenja oko svakog vrha, ispod kojeg nestaju slabe komponente.
- **Dolje lijevo:** odsječak (sivo), odsječak pomnožen prozorom (bordo) i oblik prozora
  (crtkano).
- **Dolje desno:** uvećani dio spektra oko najvišeg vrha, linearno (±`ZOOM_HZ`, a uz grube
  binove šire). Okomite crte s kružićima su binovi $f_k$. Uz nadopunu nulama puna crta prolazi
  kroz gušće točke istog spektra. Za zadanu snimku točkaste crte su prave frekvencije osnovnih
  tonova dviju žica.

Kad spektar ima više od 3000 točaka do `F_PRIKAZ` (veliki $N$ ili nadopuna), gornji graf crta
njegovu ovojnicu: u svakom malom dijelu osi samo najveću vrijednost da gusti bočni režnjevi ne
prekriju sliku. Uvećani dio i ispis računaju se iz svih točaka.

Ispod grafa ispis navodi $N$, trajanje $T$, razlučivost $\Delta f$, skalu prozora, širinu glavnog
režnja i najviši vrh. Zatim se može poslušati odsječak pomnožen prozorom: to je sve što DFT o
snimci zna.

Ako se klizači ne prikažu, pozovite funkciju izravno, npr. `pokus(8192, "Hannov", 4)`.

In [ ]:
def pokus(N=N_POCETNI, prozor=PROZOR_POCETNI, nadopuna=NADOPUNA_POCETNA):
    x = snimka[pocetak:pocetak + N]
    w = prozor_niz(prozor, N)
    f, A = spektar(x, prozor, nadopuna)
    f_b, A_b = spektar(x, prozor, 1)                     # binovi f_k (bez nadopune)
    fig = plt.figure(figsize=(12, 7.4))
    mreza = fig.add_gridspec(2, 2, width_ratios=[1, 1.35])
    ax_db = fig.add_subplot(mreza[0, :])
    ax_t = fig.add_subplot(mreza[1, 0])
    ax_z = fig.add_subplot(mreza[1, 1])

    # cijeli spektar u decibelima
    m = f <= F_PRIKAZ
    ax_db.plot(*ovojnica(f[m], 20 * np.log10(np.maximum(A[m], 1e-6))), color=TAMNA, lw=0.9)
    ax_db.set_xlim(0, F_PRIKAZ)
    ax_db.set_ylim(-110, 0)
    ax_db.set_xlabel("frekvencija $f$ / Hz")
    ax_db.set_ylabel("$20\\log_{10}A$ / dB")
    ax_db.set_title(f"jednostrani spektar: $N$ = {N}, prozor {prozor}, nadopuna ×{nadopuna}",
                    loc="left", fontsize=10)

    # odsječak, odsječak pomnožen prozorom i prozor
    t_ms = 1000 * np.arange(N) / FS
    vrh_x = np.max(np.abs(x))
    ax_t.plot(t_ms, x, color=SIVA, lw=0.5, alpha=0.6, label="odsječak $x[n]$")
    ax_t.plot(t_ms, w * x, color=BOJA, lw=0.6, label="$w[n]\\,x[n]$")
    ax_t.plot(t_ms, vrh_x * w, color=TAMNA, ls="--", lw=1.4, label="prozor (u mjerilu)")
    ax_t.set_ylim(-1.25 * vrh_x, 1.6 * vrh_x)
    ax_t.set_xlabel("vrijeme od početka odsječka / ms")
    ax_t.set_title(f"$T$ = $N/f_s$ = {N / FS * 1000:.0f} ms", loc="left", fontsize=10)
    ax_t.legend(loc="upper center", ncol=3, frameon=False, fontsize=8)

    # uvećani dio oko najvišeg vrha (za grube binove širi da se vide barem dva bina oko vrha)
    pola = max(ZOOM_HZ, 2 * FS / N)
    mb = np.abs(f_b - f_vrh) <= pola + 2 * FS / N       # i točke malo izvan okvira
    md = np.abs(f - f_vrh) <= pola + 2 * FS / N
    if nadopuna > 1:
        ax_z.plot(f[md], A[md], color=TAMNA, lw=1.2, label=f"nadopuna ×{nadopuna}")
    ax_z.vlines(f_b[mb], 0, A_b[mb], color=BOJA, lw=1.2)
    ax_z.plot(f_b[mb], A_b[mb], "o", color=BOJA, mfc="white", ms=5, mew=1.4, label="binovi $f_k$")
    if not SNIMKA:
        for fz in (F0, F0 + RASTIMANOST):
            ax_z.axvline(fz, color=SIVA, ls=":", lw=1.4)
    ax_z.set_xlim(f_vrh - pola, f_vrh + pola)
    vidljivo = np.concatenate([A[np.abs(f - f_vrh) <= pola], A_b[mb], [1e-9]])
    ax_z.set_ylim(0, 1.2 * vidljivo.max())
    ax_z.set_xlabel("frekvencija $f$ / Hz" + ("" if SNIMKA else " (točkasto: prave frekvencije)"))
    ax_z.set_ylabel("$A$")
    ax_z.set_title(f"uvećano oko najvišeg vrha, $\\Delta f$ = {FS / N:.3g} Hz".replace(".", ","),
                   loc="left", fontsize=10)
    ax_z.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    for a in (ax_db, ax_t, ax_z):
        a.xaxis.set_major_formatter(zarez)
        a.yaxis.set_major_formatter(zarez)
    fig.tight_layout()
    plt.show()
    for redak in opis(x, prozor, nadopuna):
        print(redak)
    print("Odsječak pomnožen prozorom (ono što DFT vidi):")
    display(Audio(w * x, rate=FS))


klizac = widgets.interactive(
    pokus,
    N=widgets.SelectionSlider(options=n_moguci, value=min(N_POCETNI, n_moguci[-1]),
                              description="N", continuous_update=False,
                              layout=widgets.Layout(width="520px")),
    prozor=widgets.ToggleButtons(options=PROZORI, value=PROZOR_POCETNI, description="prozor"),
    nadopuna=widgets.SelectionSlider(options=NADOPUNA_IZBOR, value=NADOPUNA_POCETNA,
                                     description="nadopuna ×", continuous_update=False,
                                     layout=widgets.Layout(width="360px")),
)
display(klizac)

## 5. Razlučivost i curenje: oba prozora i dvije duljine odsječka

Slika ispod uspoređuje oba prozora uz zadani i najveći $N$. Lijevo je uvećani dio oko najvišeg
vrha (linearno, uz nadopunu nulama ×8 da se vidi oblik vrhova), a desno širi pojas u decibelima
(binovi $f_k$). Za zadanu snimku lijevo su osnovni tonovi dviju žica, a desno i tiho brujanje.

- **Razlučivost.** Prema pravilu o razlučivanju bliskih sinusoida iz poglavlja 13 dvije sinusoide
  slične amplitude razmaknute za $\Delta$ daju zasebne vrhove u pravilu tek uz $\Delta \ge \Delta f$.
  Uz pravokutni prozor pouzdano se vide tek od $\Delta \approx 2\Delta f$, a uz Hannov prozor uz
  oko dvostruko veći razmak jer mu je glavni režanj dvostruko širi. Razlučivost poboljšava samo
  dulji odsječak; nadopuna nulama je ne mijenja.
- **Curenje.** Frekvencije zadane snimke ne leže na binovima, pa spektar curi. Uz pravokutni
  prozor rep curenja jakog vrha pada sporo i može prekriti slabu komponentu u blizini. Hannov
  prozor curi mnogo manje, pa slabu komponentu otkrije iako mu je vrh širi.

In [ ]:
nadopuna = 8
usporedba = sorted({n_zadani, n_moguci[-1]})
f_dolje, f_gore = f_vrh / 1.5, 2.3 * f_vrh      # pojas u decibelima (za vlastitu snimku mijenjajte)
fig, osi = plt.subplots(len(usporedba), 2, figsize=(12, 3.4 * len(usporedba)), squeeze=False,
                        gridspec_kw={"width_ratios": [1, 1.5]})
for red, N in zip(osi, usporedba):
    x = snimka[pocetak:pocetak + N]
    for ime, boja, crta in (("pravokutni", TAMNA, "-"), ("Hannov", BOJA, "--")):
        f, A = spektar(x, ime, nadopuna)
        m = np.abs(f - f_vrh) <= ZOOM_HZ
        red[0].plot(f[m], A[m], color=boja, ls=crta, lw=1.5, label=ime)
        f, A = spektar(x, ime, 1)                    # u decibelima: binovi f_k
        m = (f >= f_dolje) & (f <= f_gore)
        red[1].plot(f[m], 20 * np.log10(np.maximum(A[m], 1e-6)), color=boja, ls=crta, lw=1.1,
                    label=ime)
    if not SNIMKA:
        for fz in (F0, F0 + RASTIMANOST):
            red[0].axvline(fz, color=SIVA, ls=":", lw=1.4)
        red[1].axvline(F_SLABA, color=SIVA, ls=":", lw=1.4)
    naslov = f"$N$ = {N}, $T$ = {N / FS * 1000:.0f} ms, $\\Delta f$ = {FS / N:.3g} Hz"
    red[0].set_title(naslov.replace(".", ","), loc="left", fontsize=10)
    red[0].set_xlim(f_vrh - ZOOM_HZ, f_vrh + ZOOM_HZ)
    red[0].set_ylabel("$A$")
    red[1].set_xlim(f_dolje, f_gore)
    red[1].set_ylim(-100, 0)
    red[1].set_ylabel("dB")
    red[1].set_title("širi pojas u decibelima" + ("" if SNIMKA else " (točkasto: tiho brujanje)"),
                     loc="left", fontsize=10)
    red[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    for a in red:
        a.xaxis.set_major_formatter(zarez)
        a.yaxis.set_major_formatter(zarez)
for a in osi[-1]:
    a.set_xlabel("frekvencija $f$ / Hz")
fig.tight_layout()
plt.show()

## 6. Spektrogram (dodatak)

*Spektrogram* su spektri uzastopnih odsječaka snimke, poredani u vremenu: vodoravno je vrijeme,
okomito frekvencija, a tamnije je mjesto na kojem zvuk ima više snage. Svaki stupac je spektar
jednog odsječka s Hannovim prozorom. Izbor duljine odsječka je isti kompromis kao gore: kratki
odsječak prati promjene u vremenu, ali ima grubu razlučivost, a dugi ima finu razlučivost, ali
promjene u vremenu razmaže. Lijevo je odsječak od 1024 uzorka (64 ms), a desno od 16 384
(1024 ms). Pojas frekvencija isti je kao desno u odjeljku 5. Na vlastitoj snimci govora ili
glazbe spektrogram pokazuje kako se spektar mijenja u vremenu.

In [ ]:
fig, osi = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, n_seg in zip(osi, (1024, 16384)):
    if n_seg > len(snimka):
        ax.set_title(f"snimka je kraća od {n_seg} uzoraka", loc="left", fontsize=10)
        continue
    fk, tk, S = signal.spectrogram(snimka, fs=FS, window="hann", nperseg=n_seg,
                                   noverlap=n_seg * 7 // 8, mode="magnitude")
    S_db = 20 * np.log10(S / S.max() + 1e-12)
    m = (fk >= f_dolje) & (fk <= f_gore)
    ax.pcolormesh(tk, fk[m], S_db[m], cmap="gray_r", vmin=-60, vmax=0, shading="auto",
                  rasterized=True)
    ax.set_title(f"odsječak {n_seg} uzoraka ({n_seg / FS * 1000:.0f} ms)", loc="left",
                 fontsize=10)
    ax.set_xlabel("vrijeme $t$ / s")
    ax.xaxis.set_major_formatter(zarez)
    ax.grid(False)                             # mreža bi prekrila spektrogram
osi[0].set_ylabel("frekvencija $f$ / Hz")
osi[0].set_ylim(f_dolje, f_gore)
fig.tight_layout()
plt.show()

## Pokušajte sami

1. Uz pravokutni prozor i nadopunu nulama ×16 povećavajte $N$ od 2048 naviše (nadopuna samo
   pokaže oblik vrha, razlučivost ne mijenja). Od kojeg $N$ osnovni tonovi dviju žica daju dva
   vrha? Ponovite s Hannovim prozorom. Usporedite oba $N$ s pravilom o razlučivanju bliskih
   sinusoida iz poglavlja 13.
2. Uz $N = 2048$ uključite nadopunu nulama ×16. Koliko je sada točaka spektra između dvaju
   binova? Koji $N$ bez nadopune daje isti razmak točaka? Usporedite ta dva grafa u uvećanom
   dijelu.
3. Koliko je decibela tiho brujanje ispod osnovnog tona prve žice? Pročitajte to s grafa u
   decibelima i provjerite računom iz `C_SLABA` i `HARMONICI`.
4. Snimite vlastiti zvuk (zviždanje, pjevani samoglasnik, žicu gitare, zujanje ventilatora),
   spremite ga kao WAV u `moj_zvuk/` i upišite u postavke. Kolika je osnovna frekvencija? Leže li
   ostali vrhovi na njezinim višekratnicima?
5. U postavkama zadajte `RASTIMANOST = 1` i pokrenite sve ćelije. Može li ijedan $N$ s klizača
   razdvojiti žice uz pravokutni, a može li uz Hannov prozor?

---
Kod je pod licencom MIT (`LICENSE`), a tekst pod CC BY 4.0 (`LICENSE-sadrzaj.md`).